# 👥 1-Guruh Amaliyoti: Uy Narxlari Tahlili

## 🎯 Guruh maqsadi
Uy narxlariga ta'sir qiluvchi omillarni aniqlash va bashorat modeli yaratish.

## 👥 Guruh a'zolari:
- **Talaba 1**: _Ism familiya_
- **Talaba 2**: _Ism familiya_  
- **Talaba 3**: _Ism familiya_
- **Talaba 4**: _Ism familiya_

## 📋 Vazifalar taqsimoti:
- **Talaba 1**: Ma'lumotlar tahlili va tozalash
- **Talaba 2**: Korrelyatsiya tahlili va vizualizatsiya
- **Talaba 3**: Regressiya modeli va baholash
- **Talaba 4**: Bashorat va xulosalar

## ⏰ Vaqt: 90 daqiqa

In [ ]:
# Kutubxonlar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_palette("viridis")

print("🏠 1-GURUH: UY NARXLARI TAHLILI")
print("=" * 40)
print("💼 Vazifa: Ko'chmas mulk bozorini tahlil qilish")

## 📊 TALABA 1 VAZIFASI: Ma'lumotlar tahlili

### 🎯 Maqsad: Ma'lumotlarni yuklash, o'rganish va tozalash

In [ ]:
# Ma'lumotlarni yuklash
house_data = pd.read_csv('datasets/house_prices.csv')

print("🏠 UY NARXLARI MA'LUMOTLARI:")
print("=" * 30)

# TALABA 1 VAZIFALARI:
# 1. Ma'lumotlar o'lchamini ko'ring
print(f"📊 Ma'lumotlar o'lchami: {house_data.shape}")

# 2. Ustunlar haqida ma'lumot
print(f"\n📋 Ustunlar:")
print(house_data.columns.tolist())

# 3. Birinchi 5 qatorni ko'ring
print(f"\n📱 Birinchi 5 qator:")
print(house_data.head())

# 4. Ma'lumot turlari
print(f"\n🔢 Ma'lumot turlari:")
print(house_data.dtypes)

# 5. Yo'qolgan qiymatlar
print(f"\n❌ Yo'qolgan qiymatlar:")
print(house_data.isnull().sum())

In [ ]:
# TALABA 1 DAVOMI: Asosiy statistikalar

print("📈 ASOSIY STATISTIKALAR:")
print("=" * 25)
print(house_data.describe())

# Noodatiy qiymatlarni aniqlash (IQR usuli)
def find_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers

# Narx uchun outlier'lar
price_outliers = find_outliers(house_data, 'price_usd')
print(f"\n🚨 Narx bo'yicha noodatiy qiymatlar: {len(price_outliers)} ta")

# O'lcham uchun outlier'lar  
size_outliers = find_outliers(house_data, 'size_sqm')
print(f"🚨 O'lcham bo'yicha noodatiy qiymatlar: {len(size_outliers)} ta")

if len(price_outliers) > 0:
    print(f"\n📊 Eng qimmat uylar:")
    print(price_outliers.nlargest(3, 'price_usd')[['size_sqm', 'bedrooms', 'price_usd']])

## 📊 TALABA 2 VAZIFASI: Korrelyatsiya tahlili

### 🎯 Maqsad: O'zgaruvchilar orasidagi bog'lanishni aniqlash

In [ ]:
# TALABA 2 VAZIFALARI:

print("🔗 KORRELYATSIYA TAHLILI:")
print("=" * 25)

# 1. Korrelyatsiya matritsasi
numeric_cols = ['size_sqm', 'bedrooms', 'bathrooms', 'age_years', 'distance_center_km', 'price_usd']
corr_matrix = house_data[numeric_cols].corr()

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.3f', cbar_kws={'label': 'Korrelyatsiya'})
plt.title('🏠 Uy xususiyatlari korrelyatsiya matritsasi')
plt.tight_layout()
plt.show()

# 2. Narx bilan eng kuchli korrelyatsiyalar
price_corrs = corr_matrix['price_usd'].drop('price_usd').sort_values(key=abs, ascending=False)
print(f"\n💰 NARX BILAN KORRELYATSIYA:")
print("=" * 30)
for feature, corr_val in price_corrs.items():
    direction = "📈 Ijobiy" if corr_val > 0 else "📉 Salbiy"
    strength = "Kuchli" if abs(corr_val) > 0.7 else "O'rtacha" if abs(corr_val) > 0.3 else "Zaif"
    print(f"   {feature:20s}: {corr_val:6.3f} ({direction}, {strength})")

In [ ]:
# TALABA 2 DAVOMI: Scatter plotlar

# Eng muhim xususiyatlar uchun scatter plotlar
top_features = price_corrs.head(3).index.tolist()

plt.figure(figsize=(18, 5))

for i, feature in enumerate(top_features):
    plt.subplot(1, 3, i+1)
    
    plt.scatter(house_data[feature], house_data['price_usd'], alpha=0.7, s=50)
    
    # Korrelyatsiya koeffitsiyenti
    corr_val = corr_matrix.loc[feature, 'price_usd']
    
    # Regressiya chizig'i
    z = np.polyfit(house_data[feature], house_data['price_usd'], 1)
    p = np.poly1d(z)
    plt.plot(house_data[feature], p(house_data[feature]), "r--", alpha=0.8, linewidth=2)
    
    plt.xlabel(feature.replace('_', ' ').title())
    plt.ylabel('Narx (USD)')
    plt.title(f'{feature.replace("_", " ").title()}\nr = {corr_val:.3f}')
    plt.grid(True, alpha=0.3)
    plt.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

# Pearson vs Spearman taqqoslash (eng kuchli bog'lanish uchun)
best_feature = top_features[0]
pearson_corr = stats.pearsonr(house_data[best_feature], house_data['price_usd'])[0]
spearman_corr = stats.spearmanr(house_data[best_feature], house_data['price_usd'])[0]

print(f"\n🔍 {best_feature.upper()} UCHUN KORRELYATSIYA TAQQOSLASH:")
print("=" * 50)
print(f"   🔢 Pearson:  {pearson_corr:.3f}")
print(f"   📈 Spearman: {spearman_corr:.3f}")
print(f"   📊 Farq:     {abs(pearson_corr - spearman_corr):.3f}")

## 📈 TALABA 3 VAZIFASI: Regressiya modeli

### 🎯 Maqsad: Narx bashorat qilish modeli yaratish

In [ ]:
# TALABA 3 VAZIFALARI:

print("📈 REGRESSIYA MODELI YARATISH:")
print("=" * 30)

# Eng kuchli bog'lanishga ega xususiyat
best_feature = price_corrs.index[0]
print(f"🎯 Asosiy xususiyat: {best_feature}")

# 1. Ma'lumotlarni tayyorlash
X = house_data[[best_feature]]
y = house_data['price_usd']

print(f"📊 X o'lcham: {X.shape}")
print(f"📊 y o'lcham: {y.shape}")

# 2. Model yaratish va o'qitish
model = LinearRegression()
model.fit(X, y)

# 3. Model parametrlari
slope = model.coef_[0]
intercept = model.intercept_

print(f"\n📐 MODEL PARAMETRLARI:")
print("=" * 25)
print(f"   🔢 Og'ish koeffitsiyenti (a): ${slope:,.2f}")
print(f"   📍 Kesishma nuqtasi (b): ${intercept:,.2f}")
print(f"\n📝 Regressiya tenglamasi:")
if best_feature == 'size_sqm':
    print(f"   Narx = ${slope:,.0f} × O'lcham(m²) + ${intercept:,.0f}")
else:
    print(f"   Narx = ${slope:,.2f} × {best_feature} + ${intercept:,.2f}")

In [ ]:
# TALABA 3 DAVOMI: Model baholash

# 4. Bashorat qilish
y_pred = model.predict(X)

# 5. Model sifatini baholash
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = np.mean(np.abs(y - y_pred))

print(f"\n📊 MODEL SIFATI:")
print("=" * 20)
print(f"   🎯 R² (aniqlash koeffitsiyenti): {r2:.3f} ({r2*100:.1f}%)")
print(f"   📏 RMSE: ${rmse:,.0f}")
print(f"   📊 MAE: ${mae:,.0f}")

# Model sifati baholash
if r2 > 0.8:
    quality = "🌟 Juda yaxshi"
elif r2 > 0.6:
    quality = "👍 Yaxshi"  
elif r2 > 0.4:
    quality = "👌 O'rtacha"
else:
    quality = "👎 Zaif"

print(f"   📈 Model sifati: {quality}")

# 6. Model ma'nosi
print(f"\n💡 AMALIY MA'NO:")
if best_feature == 'size_sqm':
    print(f"   • Har m² uchun: ${slope:,.0f} qo'shimcha narx")
    print(f"   • 100 m² uy: ~${100 * slope + intercept:,.0f}")
elif best_feature == 'bedrooms':
    print(f"   • Har qo'shimcha xona: ${slope:,.0f} qo'shimcha narx")
else:
    print(f"   • Har birlik {best_feature}: ${slope:,.0f} narx o'zgarishi")

print(f"   • Model {r2*100:.0f}% narx farqini tushuntiradi")
print(f"   • O'rtacha xatolik: ±${rmse:,.0f}")

In [ ]:
# TALABA 3 DAVOMI: Model vizualizatsiyasi

plt.figure(figsize=(18, 5))

# 1. Regressiya chizig'i
plt.subplot(1, 3, 1)
plt.scatter(X, y, alpha=0.7, s=50, label='Uylar')
plt.plot(X, y_pred, color='red', linewidth=2, label='Regressiya chizig\'i')
plt.xlabel(best_feature.replace('_', ' ').title())
plt.ylabel('Narx (USD)')
plt.title(f'Regressiya modeli\nR² = {r2:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ticklabel_format(style='plain', axis='y')

# 2. Qoldiqlar grafigi
plt.subplot(1, 3, 2)
residuals = y - y_pred
plt.scatter(y_pred, residuals, alpha=0.7, s=50)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Bashorat qilingan narx')
plt.ylabel('Qoldiqlar (USD)')
plt.title('Qoldiqlar grafigi')
plt.grid(True, alpha=0.3)
plt.ticklabel_format(style='plain')

# 3. Haqiqiy vs Bashorat
plt.subplot(1, 3, 3)
plt.scatter(y, y_pred, alpha=0.7, s=50)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Mukammal bashorat')
plt.xlabel('Haqiqiy narx (USD)')
plt.ylabel('Bashorat qilingan narx (USD)')
plt.title('Haqiqiy vs Bashorat')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ticklabel_format(style='plain')

plt.tight_layout()
plt.show()

## 🔮 TALABA 4 VAZIFASI: Bashorat va xulosalar

### 🎯 Maqsad: Amaliy bashoratlar va tavsiyalar berish

In [ ]:
# TALABA 4 VAZIFALARI:

print("🔮 BASHORAT QILISH VA TAHLIL:")
print("=" * 35)

# 1. Test qiymatlari uchun bashorat
if best_feature == 'size_sqm':
    test_values = [60, 80, 100, 120, 150, 200]
    unit = "m²"
elif best_feature == 'bedrooms':
    test_values = [1, 2, 3, 4, 5]
    unit = "xona"
elif best_feature == 'distance_center_km':
    test_values = [2, 5, 8, 12, 15]
    unit = "km"
else:
    # Boshqa xususiyatlar uchun
    min_val = house_data[best_feature].min()
    max_val = house_data[best_feature].max()
    test_values = np.linspace(min_val, max_val, 6)
    unit = ""

print(f"💰 NARX BASHORATI ({best_feature}):")
print("=" * 40)

for value in test_values:
    predicted_price = model.predict([[value]])[0]
    
    # Ishonch oralig'i
    lower_bound = predicted_price - 1.96 * rmse
    upper_bound = predicted_price + 1.96 * rmse
    
    print(f"   🏠 {best_feature}: {value}{unit}")
    print(f"      💰 Bashorat: ${predicted_price:,.0f}")
    print(f"      📊 95% oraliq: ${lower_bound:,.0f} - ${upper_bound:,.0f}")
    
    # Ma'lumotlar oralig'ini tekshirish
    min_range = house_data[best_feature].min()
    max_range = house_data[best_feature].max()
    
    if value < min_range or value > max_range:
        print(f"      ⚠️  Ogohlantirish: Ma'lumotlar oralig'idan tashqarida!")
    print()

In [ ]:
# TALABA 4 DAVOMI: Bozor tahlili

print("📊 BOZOR TAHLILI VA XULOSALAR:")
print("=" * 35)

# 2. Bozor statistikalari
print(f"\n🏘️ BOZOR UMUMIY HOLATI:")
print("=" * 25)
print(f"   📊 O'rtacha narx: ${house_data['price_usd'].mean():,.0f}")
print(f"   📈 Mediana narx: ${house_data['price_usd'].median():,.0f}")
print(f"   💰 Eng arzon: ${house_data['price_usd'].min():,.0f}")
print(f"   💎 Eng qimmat: ${house_data['price_usd'].max():,.0f}")
print(f"   📏 O'rtacha o'lcham: {house_data['size_sqm'].mean():.0f} m²")

# 3. Narx per m²
house_data['price_per_sqm'] = house_data['price_usd'] / house_data['size_sqm']
print(f"\n💰 M² UCHUN NARX:")
print("=" * 18)
print(f"   📊 O'rtacha: ${house_data['price_per_sqm'].mean():,.0f}/m²")
print(f"   📈 Mediana: ${house_data['price_per_sqm'].median():,.0f}/m²")
print(f"   📏 Eng arzon: ${house_data['price_per_sqm'].min():,.0f}/m²")
print(f"   💎 Eng qimmat: ${house_data['price_per_sqm'].max():,.0f}/m²")

# 4. Kategoriyalarga bo'lish
def categorize_price(price):
    if price < 100000:
        return "💚 Arzon"
    elif price < 200000:
        return "💛 O'rtacha"
    elif price < 300000:
        return "🧡 Qimmat"
    else:
        return "❤️ Juda qimmat"

house_data['price_category'] = house_data['price_usd'].apply(categorize_price)
price_distribution = house_data['price_category'].value_counts()

print(f"\n🏷️ NARX KATEGORIYALARI:")
print("=" * 22)
for category, count in price_distribution.items():
    percentage = count / len(house_data) * 100
    print(f"   {category}: {count} ta ({percentage:.1f}%)")

In [ ]:
# TALABA 4 DAVOMI: Yakuniy xulosalar va tavsiyalar

print(f"\n📋 YAKUNIY XULOSALAR:")
print("=" * 25)

print(f"\n🎯 ASOSIY NATIJALAR:")
print(f"   • Eng muhim omil: {best_feature.replace('_', ' ').title()}")
print(f"   • Korrelyatsiya kuchi: {price_corrs.iloc[0]:.3f} ({'Kuchli' if abs(price_corrs.iloc[0]) > 0.7 else 'O\'rtacha'})")
print(f"   • Model aniqlik: {r2*100:.0f}% (R²)")
print(f"   • Bashorat xatoligi: ±${rmse:,.0f}")

if best_feature == 'size_sqm':
    print(f"\n🏠 XARIDORLAR UCHUN MASLAHATLAR:")
    print(f"   • Har m² uchun ~${slope:,.0f} to'lanadi")
    print(f"   • 100 m² uy: ~${100 * slope + intercept:,.0f}")
    print(f"   • O'lcham oshirish samarali investitsiya")
elif best_feature == 'distance_center_km':
    print(f"\n🚗 JOYLASHUV MASLAHATLAR:")
    if slope < 0:
        print(f"   • Markazga yaqinroq - qimmatroq")
        print(f"   • Har km uchun ~${abs(slope):,.0f} tejash")
    else:
        print(f"   • Markazdan uzoqroq - qimmatroq")
        print(f"   • Har km uchun ~${slope:,.0f} qo'shimcha")

print(f"\n💡 INVESTORLAR UCHUN:")
print(f"   • Model {r2*100:.0f}% narx o'zgarishini tushuntiradi")
print(f"   • {100-r2*100:.0f}% boshqa omillarga bog'liq")
print(f"   • {best_feature.replace('_', ' ').title()} - eng muhim investitsiya omili")

print(f"\n⚠️ CHEKLOVLAR:")
print(f"   • Faqat bitta omil hisobga olingan")
print(f"   • Ma'lumotlar oralig'idan tashqarida xavfli")
print(f"   • Bozor o'zgarishlari hisobga olinmagan")
print(f"   • Joylashuv, ta'mirlash holati e'tiborga olinmagan")

print(f"\n🎯 GURUH XULOSASI:")
print(f"   Model uy narxlarini {r2*100:.0f}% aniqlik bilan bashorat qila oladi.")
print(f"   {best_feature.replace('_', ' ').title()} - narxga eng kuchli ta'sir ko'rsatuvchi omil.")
print(f"   Real hayotda ko'proq omillarni hisobga olish kerak.")

## 📤 Guruh hisoboti

### 👥 Guruh a'zolari hissasi:
- **Talaba 1**: Ma'lumotlar tahlili, tozalash va asosiy statistikalar
- **Talaba 2**: Korrelyatsiya matritsasi, visualizatsiya va bog'lanish tahlili  
- **Talaba 3**: Regressiya modeli yaratish, o'qitish va sifat baholash
- **Talaba 4**: Bashorat qilish, bozor tahlili va yakuniy xulosalar

### 🎯 Asosiy natijalar:
1. **Eng muhim omil**: [Talaba 2 tomonidan aniqlangan]
2. **Model sifati**: [Talaba 3 tomonidan baholangan]
3. **Amaliy tavsiyalar**: [Talaba 4 tomonidan tayyorlangan]

### 💡 Guruh tavsiyalari:
- [Bu yerda guruh umumiy xulosasini yozing]
- [Model yaxshilash uchun takliflar]
- [Kelajakdagi tadqiqotlar uchun g'oyalar]

### 📊 Taqdimot uchun asosiy grafiklar:
1. Korrelyatsiya matritsasi
2. Eng kuchli bog'lanish scatter plot
3. Regressiya modeli va qoldiqlar
4. Bashorat natijalari

---

**⏰ Taqdimot vaqti: 10 daqiqa**  
**👥 Har bir a'zo: 2-3 daqiqa**